# NB11 - Project Blueprint

Status: **plan only**. No code. The functions are empty on purpose. They are there to show
the shape of what we intend to build so we can argue about it before writing it.

Last updated: 2026-08-04

## Read this first (this gives you a base of understanding)


1. We built a lot of separate pipelines. None of them talk to each other.
2. Nobody can answer "what do we know about company 09857485" without opening six notebooks.
3. So we build one small layer that joins their outputs into a single file.
4. **The thing we are proving is the joined file, not the screen.** The dashboard is just a way to
   look at it.
5. Nobody rewrites anything. Each pipeline stays where it is and gains a thirty line adapter.
6. The dashboard reads a prebuilt file. It never calls an API.

## 1. Distribution


Between the four of us we built roughly twenty notebooks across eight branches.

| who | built |
|---|---|
| Vishal | Gazette crawl and distress features, news and sentiment |
| Samuel | company spine, name to number matcher, contracts, hiring, property, trade marks |
| Viktor | monthly company panel, four trained models |
| Sneha | lender relationships, director changes |

All of it works. Each one saves its results in its own place, in its own shape, with its own idea of
what a company key looks like. Some are CSV, some DuckDB, at least one was never saved at all.

So this question currently takes an hour to answer, and only one of us can do it:

> What do we know about company 09857485?

That is the only problem this work solves. Not data collection, we have the data. Not modelling,
Viktor and Sneha have that. Not coverage, we know it is low and that is a finding. **A joining
problem.**

Lloyds asked whether public data can show which companies are about to need a bank. A method for
linking the sources is one of the four things the brief asks for, and a method is not a paragraph in
a report. It is a working thing that takes a company and returns everything we know about it.

## 2. The idea

Every pipeline keeps working exactly as it does today. We add one thin translation step after each
one, so everything lands in the same place in the same shape. Then we put a screen on top.

```
    your notebook  ->  adapter  ->  the store  ->  the dashboard
    (unchanged)        (~30 lines)  (one file)     (reads only)
```

The important word is **unchanged**. If your notebook saves a CSV today, it carries on saving that
CSV. The adapter is separate code, owned by you, that reads your CSV and writes rows into the store.

**you miight ask...Why not rebuild everything into one clean codebase?**

- The work is already done. The Gazette crawl alone is 284,230 notices from an overnight run.
- Each pipeline is somebody's dissertation material. Merging them makes contributions invisible.
- We have about a month. Four adapters is a week. Twenty rewrites is not.
- It isolates failure. If the news adapter breaks, the Gazette panel still works.

**The harmonisation layer is three boring things:** one agreed row shape, one adapter per source,
and one build script. That is all. If it starts growing business logic, that logic belongs back in
whichever pipeline owns it.

## 3. The five rules

Every one of these exists because we have already been bitten by it.

| # | rule | why |
|---|---|---|
| 1 | Join on company number, never name | Names collide, change, and are punctuated differently in every source. One shared cleaning function, imported by everyone. |
| 2 | Every signal carries a real date, no nulls | Without a date we cannot draw a timeline, and we cannot stop the fact leaking into Viktor's panel, which only allows facts knowable at the time. |
| 3 | Absence is zero, not blank | Most UK companies have no news and no notices. "Nothing found" is a real answer and the clearest finding of the project. |
| 4 | Confidence travels with the fact | Only 57% of Gazette notices carry a company number, and name plus postcode matching is about 92% right. A shaky match must not look certain. |
| 5 | The store is descriptive, not predictive | It describes companies as of a fixed date. Fine for a report a person reads. Not valid as model training input. |

`SNAPSHOT_DATE = 2026-06-30`, pinned, written into every table, shown on every screen.

## 4. How data flows

Two completely separate times. Keeping them apart is the most important decision in this document.

**Build time.** Runs on a laptop, takes minutes, happens when we choose. All the work and all the
API calls happen here.

```
  NB10  gazette notices + features --> gazette_adapter()  --\
  NB09  guardian + FinBERT ---------> news_adapter()  ------+--> builder --> company_store.duckdb
  NB06  contracts finder -----------> contracts_adapter() --/
  SHAP  scored companies -----------> scores_adapter()  ---/
```

**Read time.** Runs when someone types a name. Under a second. Touches no APIs, no CSVs, no
notebooks. Only the store file.

```
  user types "Example Trading"
        |
        v
  find_company()  ------> more than one match? show a list, let them pick
        |
        v
  company_number = "09857485"
        |
        v
  get_company_view()  ---> one CompanyView object ---> the screen
```

Why split it this way:

- A live demo depending on four external APIs will fail in front of the team at least once.
- NewsAPI allows 100 requests a day. GDELT banned our IP once for asking too fast.
- The Gazette side is an overnight crawl, not a request time operation.
- An examiner needs the same numbers next month. A prebuilt store gives that.

A "refresh this company" button is possible later. Not in the MVP.

## 5. What the store looks like

One DuckDB file, five tables. We adopt the schema Samuel already built in `05_spine_crosswalk.ipynb`
rather than inventing one.

```
company_store.duckdb
  +-- companies        one row per company. the spine.
  +-- identifiers      names and codes that point at a company.
  +-- signals          one row per event. long and thin. the evidence.
  +-- company_signals  one row per company. wide. the summary.
  +-- scores           one row per company. the model output.
```

Plus the wide tables that rich sources produce and that do not fit a thin row, kept whole, for
example `gazette_features` at 106,664 rows by 56 columns.

Both shapes exist for a reason. A contract award is one fact on one date, so a thin row fits. The
Gazette work produces 52 engineered features per company, and squashing those into one value per row
would throw the work away.

### The agreed row shape

This is the one thing everybody has to produce. Eight columns, and they never change.

| column | meaning | example |
|---|---|---|
| `company_number` | the key, always | `09857485` |
| `signal_type` | which kind of thing happened | `gazette_petition` |
| `signal_date` | when it happened, never null | `2026-05-09` |
| `value` | a number if there is one | `1` |
| `detail` | short text shown on screen | `Petition to wind up` |
| `source` | which pipeline produced this row | `gazette` |
| `confidence` | 1.0 exact, lower if fuzzy | `1.00` |
| `retrieved_at` | when we pulled it | `2026-06-30` |

Filled in for one company. These rows come from four different people's notebooks and sit next to
each other happily.

| company_number | signal_type | signal_date | value | detail | source | confidence |
|---|---|---|---|---|---|---|
| 09857485 | contract_win | 2025-03-14 | 240000 | Grounds maintenance, Bristol CC | contracts | 1.00 |
| 09857485 | hiring_post | 2025-09-02 | 3 | 3 open roles, warehouse | adzuna | 0.80 |
| 09857485 | news_article | 2026-04-18 | -0.62 | Supplier disputes late payment | news | 0.95 |
| 09857485 | gazette_petition | 2026-05-09 | 1 | Petition to wind up | gazette | 1.00 |
| 09857485 | gazette_admin | 2026-06-02 | 1 | Notice of administrator appointment | gazette | 1.00 |

Read down the date column. That is the timeline, and it is the one view no single branch can produce
today.

### The unified company table

Built by aggregating `signals` up to one row per company, then left joining onto the full spine so
companies with nothing get zeros instead of disappearing. The dashboard tiles read this.

| company_number | snapshot_date | gaz_notice_count | gaz_worst_severity | gaz_latest_date | news_article_count | news_sentiment | contract_count | contract_value_total | hiring_count | has_any_signal | invisible |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 09857485 | 2026-06-30 | 2 | formal_insolvency | 2026-06-02 | 2 | -0.62 | 1 | 240000 | 3 | True | False |
| 00445790 | 2026-06-30 | 0 | none | NULL | 47 | 0.11 | 12 | 8400000 | 210 | True | False |
| 12345678 | 2026-06-30 | 0 | none | NULL | 0 | NULL | 0 | 0 | 0 | False | True |

Two things to notice.

- **Row 3 is the normal case.** Most companies in the book look like that. The screen has to handle
  it well, because it is not an edge case.
- Counts default to `0`, measurements default to `NULL`. "No sentiment" and "neutral sentiment" are
  not the same thing.

### The evidence tables

These sit behind the panels and the links. Rich sources keep their natural shape.

**gazette_evidence**

| company_number | notice_type | notice_family | severity_tier | notice_date | url | confidence |
|---|---|---|---|---|---|---|
| 09857485 | Petition to wind up | insolvency | formal_insolvency | 2026-05-09 | thegazette.co.uk/notice/... | 1.00 |
| 09857485 | Appointment of administrator | insolvency | formal_insolvency | 2026-06-02 | thegazette.co.uk/notice/... | 1.00 |

**news_evidence**

| company_number | headline | publication | published_date | sentiment_score | sentiment_label | verified | confidence |
|---|---|---|---|---|---|---|---|
| 09857485 | Supplier disputes late payment | Guardian | 2026-04-18 | -0.62 | negative | True | 0.95 |
| 09857485 | Local firm cuts warehouse shifts | Guardian | 2026-05-30 | -0.41 | negative | True | 0.90 |

The `verified` column matters. On the 96 company test sample, 5 companies returned a raw hit and 0
survived verification, because every one turned out to be a different company with a similar name.
Unverified rows are kept so the finding is inspectable, but never counted.

**contract_evidence**

| company_number | title | buyer | award_value | published_date | confidence |
|---|---|---|---|---|---|
| 09857485 | Grounds maintenance framework | Bristol City Council | 240000 | 2025-03-14 | 1.00 |

Note **published** date, not signature date. Viktor's pipeline gates on publication because that is
the first moment the fact was knowable. We follow the same rule so both sides stay compatible.

**scores**, the only thing we need from the modelling side, as a plain file.

| company_number | lending_3m | insolvency_6m | voluntary_exit_6m | growth_12m | model_run_date |
|---|---|---|---|---|---|
| 09857485 | 0.031 | 0.940 | 0.220 | 0.004 | 2026-07-28 |
| 12345678 | 0.052 | 0.003 | 0.090 | 0.021 | 2026-07-28 |

We display these as signal indices, never as predictions. There is no labelled ground truth behind
them.

## 6. What the dashboard shows

It does three things and nothing else: turn what the user typed into a company number, read one row
from each table, draw it. No matching logic, no scoring, no thresholds, no API calls.

Two ways in. The search box is obvious. The ranked list is the one that tells the Lloyds story,
because it is what a relationship manager would open on a Monday.

```
 ===========================================================================
  EXAMPLE TRADING LTD                                    All data as of
  09857485  ·  Active  ·  Incorporated 2015-11-03         30 June 2026
  Unit 4, Feeder Road, Bristol, BS2 0SB
  Manufacturing  ·  Small
 ===========================================================================

  SIGNAL INDICES  (from the structured model, not predictions)
    Insolvency, 6 month index      0.94    high
    Lending readiness, 3 month     0.03    low
    Growth, 12 month               0.00    low

 ---------------------------------------------------------------------------
  TIMELINE
    2025-03-14   contracts   Won: Grounds maintenance, Bristol CC   240,000
    2025-09-02   hiring      3 open roles, warehouse
    2026-04-18   news        Supplier disputes late payment         negative
    2026-05-09   gazette     Petition to wind up
    2026-06-02   gazette     Notice of appointment of administrator

 ---------------------------------------------------------------------------
  GAZETTE              2 notices, most recent 2026-06-02
  NEWS                 2 verified articles, average tone negative
  PUBLIC CONTRACTS     1 award, 240,000 total
  HIRING               3 adverts in the last 12 months
  PROPERTY             nothing found
  TRADE MARKS          nothing found

 ---------------------------------------------------------------------------
  EVIDENCE
    [ Gazette notice 4512338 ]  [ Guardian article, 18 Apr ]
    [ Gazette notice 4571902 ]  [ Contracts Finder record ]
 ===========================================================================
```

### The empty page, which is the normal one

Gazette distress touches 0.075% of the active book. Verified news touches close to zero. Most
searches look like this, so we design this screen first and make it confident rather than broken.

```
 ===========================================================================
  QUIET SERVICES LTD                                     All data as of
  12345678  ·  Active  ·  Incorporated 2019-04-22         30 June 2026
  14 Kirkstall Road, Leeds, LS3 1JL
  Technology  ·  Micro
 ===========================================================================

  SIGNAL INDICES
    Insolvency, 6 month index      0.003   low
    Lending readiness, 3 month     0.052   low
    Growth, 12 month               0.021   low

 ---------------------------------------------------------------------------
  NO EXTERNAL SIGNALS FOUND

  We searched all seven sources and found nothing for this company.
  That is a result, not an error.

    Gazette        checked, nothing found
    News           checked, nothing found
    Contracts      checked, nothing found
    Hiring         checked, nothing found
    Property       checked, nothing found
    Trade marks    checked, nothing found
    Grants         checked, nothing found
 ===========================================================================
```

The wording is deliberate. "Checked, nothing found" is different from "no data", which sounds like
we failed to look.

## 7. The shape of the code

Nothing below is implemented. Names and signatures are the point. Bodies are empty so we can argue
about the structure first.

In [ ]:
from dataclasses import dataclass, field
from datetime import date
from typing import Any

SNAPSHOT_DATE = date(2026, 6, 30)      # pinned, never computed from today()
STORE_PATH = "data/processed/company_store.duckdb"


def clean_company_number(raw: Any) -> str | None:
    """Turn anything that looks like a company number into the canonical form.

    The single most important function in the project, because every join in the
    store depends on both sides agreeing. It lives in one place and every adapter
    imports it. Nobody writes their own version.

    Must cope with, at least:
      - integers that lost leading zeros in Excel or pandas   (445790)
      - values already correct                                ("00445790")
      - Scottish and Northern Irish prefixes                  ("SC123456")
      - whitespace, including our CSV column " CompanyNumber"
      - None, empty string, and the literal text "nan"

    Returns 8 characters, or None if the input cannot be a company number.
    Returning None rather than raising is deliberate: a bad row should be dropped
    and counted, not crash a build of two million rows.
    """
    ...

In [ ]:
@dataclass
class Signal:
    """One fact, about one company, on one date.

    The agreed contract, as a class so adapters fail loudly if they forget a
    field. Every adapter returns a list of these. Nothing extra is allowed here.
    If a source has richer information than fits, it goes in that source's own
    evidence table and this row points at it.
    """

    company_number: str    # cleaned, 8 chars, never None
    signal_type: str       # from SIGNAL_TYPES below
    signal_date: date      # never None, see rule 2
    value: float | None    # a number if there is one
    detail: str            # short text, shown on screen as written
    source: str            # which pipeline produced this
    confidence: float      # 1.0 exact, lower if the match was fuzzy
    retrieved_at: date     # when the underlying fetch happened


# Agreed once, then frozen. Adding a type is one line here plus a screen label.
SIGNAL_TYPES = [
    "gazette_petition", "gazette_administration", "gazette_liquidation",
    "gazette_strike_off", "news_article", "contract_win", "hiring_post",
    "property_holding", "trademark_filing", "research_grant",
    "charge_registered", "charge_satisfied",
]

In [ ]:
class SourceAdapter:
    """What every adapter looks like.

    Reads a file a pipeline already produces and returns rows in the agreed
    shape. Never fetches anything, never calls an API, never modifies the
    pipeline it reads from. A class rather than a loose function buys one thing:
    the builder can loop over adapters without knowing what any of them do.
    """

    name: str = "unnamed"      # goes into the source column
    input_path: str = ""       # the file this adapter reads

    def load(self):
        """Read the file the pipeline already saved. No cleverness here."""
        ...

    def to_signals(self, raw) -> list[Signal]:
        """Turn loaded data into Signal rows.

        All per source knowledge lives here. Dropping rows without a usable date
        happens here, and the count of dropped rows is reported, never hidden.
        """
        ...

    def to_evidence(self, raw):
        """Optional. Rich sources also return their own wide table."""
        ...

    def report(self) -> dict:
        """Numbers printed after every build, so problems are visible.

        Rows in, rows out, rows dropped and why, date range, share of rows below
        confidence 1.0. If these move unexpectedly, something upstream changed.
        """
        ...


class GazetteAdapter(SourceAdapter):
    """Vishal. Reads nb10_gazette_notices.csv (284,230 notices) and
    nb10_gazette_company_features.csv (106,664 companies x 56 columns).

    Produces signals, gazette_evidence, and passes the feature table through
    unchanged.

    Known limit to carry through, not hide: only about 57% of notices carry a
    company number. The rest are name only and excluded for now, roughly 85,000
    distinct unmatched names. That number goes in the build report and the
    write up.
    """


class NewsAdapter(SourceAdapter):
    """Vishal. Reads nb09_guardian_signals.csv.

    Only verified articles become signals. Unverified rows stay in the evidence
    table with verified=False so the finding is inspectable, but are never
    counted.
    """


class ContractsAdapter(SourceAdapter):
    """Samuel. Reads the Contracts Finder output.

    Dates use publication, not signature, to match Viktor's rule. Contracts
    Finder has been built three times across the branches. Pick one, note the
    others as duplicates, move on.
    """


class ScoresAdapter(SourceAdapter):
    """Viktor and Sneha's output, read as a plain file.

    The only dependency we have on the modelling side, and it is a file, not an
    import. That is on purpose: both halves carry on independently until the day
    we join them.
    """

In [ ]:
def build_store(adapters: list[SourceAdapter], out_path: str = STORE_PATH):
    """Run every adapter and write one store file.

    In order:
      1. Load the spine from Samuel's crosswalk. This defines which companies
         exist. Anything not on the spine is dropped and counted, never silently
         ignored.
      2. Call each adapter, collect signal rows and evidence tables.
      3. Stack all signal rows into one table.
      4. Aggregate signals up to one row per company, then LEFT JOIN onto the
         full spine so companies with nothing get zeros. This step is rule 3.
      5. Attach the model scores.
      6. Write everything to a fresh DuckDB file.
      7. Print the build report.

    Rebuilds from scratch every time. No incremental update, no migration, no
    state. Faster to reason about and impossible to get subtly wrong.
    """
    ...


def build_report(store_path: str = STORE_PATH) -> dict:
    """The numbers we look at after every build.

    Companies on the spine, signals per source, companies with at least one
    signal, companies with nothing from anywhere, share below confidence 1.0,
    earliest and latest signal date.

    Read every single time. It is how we notice that a source silently produced
    zero rows because a file moved.
    """
    ...

In [ ]:
@dataclass
class CompanyView:
    """Everything the screen needs about one company, already assembled.

    The dashboard receives one of these and draws it. It does no lookups of its
    own, so the screen can be swapped for a different one, or for a printed
    report, without touching anything else.
    """

    profile: dict                      # name, number, status, address, sector
    scores: dict                       # the model indices
    summary: dict                      # the wide company_signals row
    timeline: list = field(default_factory=list)
    gazette: list = field(default_factory=list)
    news: list = field(default_factory=list)
    contracts: list = field(default_factory=list)
    snapshot_date: date = SNAPSHOT_DATE

    def is_empty(self) -> bool:
        """True when no source returned anything.

        Switches the screen to the "checked, nothing found" layout. A first
        class case, not an error path.
        """
        ...


def find_company(text: str) -> list[dict]:
    """Turn what the user typed into candidate companies.

    Owned by Samuel, who already built the name plus postcode matcher.

    Accepts a number or a name. Always returns a list, even when there is one
    obvious answer, because "Tesco" matches many companies and the user has to
    pick. Each candidate carries a confidence so weak matches look weak.
    Returns an empty list when nothing matches, and the screen says so plainly.
    """
    ...


def get_company_view(company_number: str) -> CompanyView:
    """Read one company out of the store and return it assembled.

    Reads only. Opens the file, runs a handful of selects, closes it. Well under
    a second. If this ever needs a network connection, the design has gone wrong.
    """
    ...


def render_company_page(view: CompanyView) -> str:
    """Draw the page. Owned by Vishal.

    Takes a CompanyView, returns something a person can read. Makes no decisions
    about what the data means. Must always show three things: the snapshot date
    so nobody thinks this is live, the confidence on any uncertain fact, and
    "checked, nothing found" rather than a blank space.
    """
    ...

## 8. Who owns what

Split by type of source, which is what Viktor asked for. Once the shape is agreed, neither of us
needs to read the other's code.

**Samuel**

- Identity: `find_company()`, the spine, the crosswalk, the name plus postcode matcher. This is the
  front door and he is the only person who has built it.
- Government and transactional sources: Contracts Finder, Adzuna hiring, Land Registry, IPO trade
  marks, financial mood. One adapter each.
- The profile half of the screen: header, address, size, sector, summary tiles.
- `build_store()`, because it sits closest to the spine.

**Vishal**

- Text and media sources: Gazette (notices, features, severity), news and sentiment. One adapter each.
- The signal contract: the `Signal` shape, `SIGNAL_TYPES`, and `clean_company_number()`. Keeping
  these honest across sources is a job, not a one off decision.
- The evidence half of the screen: timeline, notice and article panels, provenance on every fact.
- The build report, because it is mostly a data quality question.

**Viktor and Sneha** are not part of this and nothing here asks them to change anything. We need one
thing: their scored output saved as a file with `company_number`, one column per target, and the run
date. When they save a newer file we rebuild and pick it up.

**Agreed by all of us before any code:** the eight column signal shape, the type vocabulary, the
company number cleaning rule, the snapshot date. Those four things are the interface. Everything
else is private to whoever owns it.

## 9. Order of work

Deliberately small steps, so a mistake is found while it is still cheap.

| step | what | who |
|---|---|---|
| 0 | Agree this document. Settle the four shared things and the open questions. | all |
| 1 | Write `clean_company_number()` and its tests. Half a day. Everything depends on it. | Vishal |
| 2 | Build an empty store on Samuel's schema, spine loaded, nothing else. | Samuel |
| 3 | **One adapter each.** Vishal does Gazette, Samuel does Contracts. Then look at the result together. | both |
| 4 | Add the news adapter and the scores file. Three people's work, both halves joined. | both |
| 5 | The screen. Two layouts only: a company with signals, and a company with nothing. | Vishal |
| 6 | Pick demo companies from the 1,033 company active watchlist, plus one with nothing found. | both |

Step 3 is the checkpoint. If the shape is wrong we find out there, having written about sixty lines
of code between us.

**For the MVP we take four sources only:** Gazette and News (Vishal), Contracts (Samuel), and the
model scores (Viktor). Three people, both halves of the project, enough to prove the design. The
fifth adapter teaches us nothing new, so it waits.

**Not Tesco for the demo.** It has coverage no company in the target book has, so it proves nothing
and invites the obvious question about whether this works on real prospects.

**Done means:** someone who is not us clones the repo, runs one script, and gets the same page for
the same company number. No API keys, no overnight crawl, no manual steps. If that is true, the
harmonisation claim is proven and everything after it is presentation.

## 10. Open questions and risks

**Settle this one first.** Does Viktor's 2,038,130 company panel keep companies after they stop
being Active? The Gazette feature table covers 106,664 companies, but only 1,033 of them are in the
1.37 million active universe, because a company with a winding up petition is on its way out of
Active status. If the panel is active only, our strongest source fires on about one deep dive in a
thousand. Ask Viktor before we build anything.

**Still open**

- Which company list is the master list. Three exist and disagree: Samuel's spine, the SHAP feature
  matrix at 1,372,321, and the panel at 2,038,130. Until one is picked, every join quietly drops rows.
- The spine size itself. One reading put it at 869,043 companies, another note put signals coverage
  at 45,584. Probably counting different things, but it needs confirming.
- Which sentiment method is the record. FinBERT on one side, VADER and TextBlob on the other.
  FinBERT is the current choice.

**Risks we already know about**

- Not everything is on disk. At least one notebook produced results only in memory, capped at 100
  rows. If it is not a file it cannot be adapted.
- Notebooks are not importable. Adapters read saved outputs. Nobody executes a notebook from the
  builder.
- The data files are not in git. We need either a small store file committed or a script that
  rebuilds it, or nobody outside the team can run the demo and the "done" test fails.
- Identity resolution is on the critical path and is one person's work. Mitigation: accept a raw
  company number as a first class input from day one, so the screen is never blocked on the matcher.
- Static versus time aware. The store describes companies as of the snapshot date. Correct for a
  report a person reads, wrong as model training input. We label it and do not hand it over as
  features.
- Scope creep towards live calls. Every live API call on the read path is a new way for the demo to
  fail. The answer is no for the MVP.